# 피싱 탐지 레이더 아이콘 만들기 (틸론 GPU)

위에서부터 칸을 하나씩 선택하고 **Shift + Enter**를 누르면 실행됩니다.

만들 그림 5종 (파일 이름은 바꾸지 마세요)

| 파일 이름 | 쓰는 곳 |
|---|---|
| radar-shield.png | 서비스 아이콘 (맨 위 로고) |
| analyzing.png | 분석 중 화면 |
| risk.png | 위험·주의 결과 |
| clear.png | 뚜렷한 위험 신호 없음 결과 |
| uncertain.png | 판단 어려움 결과 |

※ 인터넷에서 그림 생성 모델(수 GB)과 배경 제거 모델을 내려받습니다. 처음 한 번은 시간이 걸립니다.

In [ ]:
# 1) GPU 확인
!nvidia-smi

In [ ]:
# 2) 필요한 프로그램 설치 (몇 분 걸릴 수 있음). %pip 는 지금 노트북 환경에 바로 설치합니다.
%pip install diffusers transformers accelerate safetensors "rembg[cpu]" onnxruntime pillow matplotlib

In [ ]:
# 3) 그림 생성 모델 불러오기: GPU 메모리가 12GB 이상이면 SDXL, 아니면 가벼운 SD 1.5 를 씁니다.
import torch
from diffusers import StableDiffusionXLPipeline, StableDiffusionPipeline

assert torch.cuda.is_available(), "GPU 를 찾지 못했습니다. GPU 가 있는 환경에서 실행해 주세요."
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU 메모리: {vram_gb:.1f} GB")

if vram_gb >= 12:
    MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
    pipe = StableDiffusionXLPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    SIZE = 1024
else:
    MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=torch.float16, use_safetensors=True)
    SIZE = 512
pipe = pipe.to("cuda")
print("사용 모델:", MODEL)

In [ ]:
# 4) 그림 설명(프롬프트). 같은 캐릭터·같은 그림체가 되도록 공통 설명을 붙입니다.
#    색: 기본 네이비·파랑 / 위험 주황·빨강 / 위험 신호 없음 초록 / 판단 어려움 회색. 그림 안에 글자는 넣지 않습니다.
CHARACTER = "a cute round friendly robot mascot with a small radar antenna on its head, navy blue and bright blue body"
STYLE = ("flat vector illustration, simple geometric shapes, bold clean outlines, soft shading, centered, "
         "single subject, plain white background, no text, high quality app icon style")
NEGATIVE = ("text, letters, words, numbers, watermark, signature, blurry, photo, realistic, 3d render, "
            "busy background, multiple characters, cropped, frame, border")

PROMPTS = {
    "radar-shield": "a security shield emblem with a radar screen in the center, navy blue and bright blue, " + STYLE,
    "analyzing": CHARACTER + ", holding a magnifying glass and examining closely, focused expression, " + STYLE,
    "risk": CHARACTER + ", worried expression, holding an orange and red warning triangle sign, " + STYLE,
    "clear": CHARACTER + ", calm gentle smile, next to a green check mark, " + STYLE,
    "uncertain": CHARACTER + ", puzzled expression, scratching its head, gray question mark beside it, " + STYLE,
}
print("준비된 그림:", list(PROMPTS))

In [ ]:
# 5) 후보 그림 만들기: 그림마다 4장씩 만들고 번호(0~3)를 붙여 보여 줍니다.
#    마음에 드는 게 없으면 이 칸을 다시 실행하면 다른 후보가 나옵니다(SEED_START 숫자를 바꿔도 됩니다).
import matplotlib.pyplot as plt

CANDIDATES_PER_ICON = 4
SEED_START = 100
candidates = {}

for name, prompt in PROMPTS.items():
    images = []
    for k in range(CANDIDATES_PER_ICON):
        generator = torch.Generator("cuda").manual_seed(SEED_START + k)
        image = pipe(prompt=prompt, negative_prompt=NEGATIVE, width=SIZE, height=SIZE,
                     num_inference_steps=30, guidance_scale=7.0, generator=generator).images[0]
        images.append(image)
    candidates[name] = images
    fig, axes = plt.subplots(1, CANDIDATES_PER_ICON, figsize=(4 * CANDIDATES_PER_ICON, 4))
    for k, (ax, image) in enumerate(zip(axes, images)):
        ax.imshow(image)
        ax.set_title(f"{name}  #{k}")
        ax.axis("off")
    plt.show()
SEED_START += CANDIDATES_PER_ICON

## 6) 마음에 드는 번호 고르기

아래 칸의 숫자(0~3)를 위에서 고른 후보 번호로 바꾼 뒤 실행하세요.
배경을 지우고 투명 배경 1024×1024 PNG 로 저장합니다.

In [ ]:
CHOICES = {
    "radar-shield": 0,
    "analyzing": 0,
    "risk": 0,
    "clear": 0,
    "uncertain": 0,
}

import os
from PIL import Image, ImageDraw

OUT_DIR = "phishing_radar_icons"
os.makedirs(OUT_DIR, exist_ok=True)
FINAL = 1024
MARGIN = 0.06  # 가장자리 여백 비율

def remove_white_background(image, thresh=40):
    """rembg 가 없을 때: 그림 가장자리에서 이어진 흰 배경만 투명하게 바꾼다(캐릭터 안의 흰색은 남김)."""
    rgb = image.convert("RGB")
    marker = (255, 0, 255)
    w, h = rgb.size
    for xy in [(0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1)]:
        if rgb.getpixel(xy) != marker:
            ImageDraw.floodfill(rgb, xy, marker, thresh=thresh)
    rgba = rgb.convert("RGBA")
    rgba.putdata([(r, g, b, 0) if (r, g, b) == marker else (r, g, b, 255) for (r, g, b, a) in rgba.getdata()])
    return rgba

try:
    from rembg import remove
    def cut_out(image):
        return remove(image).convert("RGBA")
    print("배경 제거: rembg 사용")
except ImportError:
    cut_out = remove_white_background
    print("배경 제거: rembg 가 없어 흰 배경만 지우는 방식으로 진행합니다")

def to_transparent_square(image):
    cut = cut_out(image)                         # 배경 지우기
    box = cut.getbbox()                          # 그림이 있는 부분만 남기기
    if box:
        cut = cut.crop(box)
    side = int(max(cut.size) * (1 + 2 * MARGIN))
    canvas = Image.new("RGBA", (side, side), (0, 0, 0, 0))
    canvas.paste(cut, ((side - cut.width) // 2, (side - cut.height) // 2), cut)
    return canvas.resize((FINAL, FINAL), Image.LANCZOS)

saved = {}
for name, pick in CHOICES.items():
    result = to_transparent_square(candidates[name][pick])
    path = os.path.join(OUT_DIR, name + ".png")
    result.save(path, optimize=True)
    saved[name] = result
    print(f"저장: {path}  ({os.path.getsize(path) / 1024:.0f} KB)")

# 투명 배경이 잘 됐는지 체크무늬 위에 놓고 보여 줍니다.
def checker(size=FINAL, cell=64):
    board = Image.new("RGBA", (size, size), (255, 255, 255, 255))
    for y in range(0, size, cell):
        for x in range(0, size, cell):
            if (x // cell + y // cell) % 2:
                board.paste((220, 220, 220, 255), (x, y, x + cell, y + cell))
    return board

fig, axes = plt.subplots(1, len(saved), figsize=(4 * len(saved), 4))
for ax, (name, image) in zip(axes, saved.items()):
    preview = checker()
    preview.alpha_composite(image)
    ax.imshow(preview)
    ax.set_title(name)
    ax.axis("off")
plt.show()

In [ ]:
# 7) zip 파일로 묶기: 왼쪽 파일 목록에서 phishing_radar_icons.zip 을 오른쪽 클릭 → Download 하세요.
import shutil
zip_path = shutil.make_archive("phishing_radar_icons", "zip", OUT_DIR)
print("만든 파일:", zip_path)